# 현재 7클래스 모델 추가 학습 — 저화질 적응 / 색상 유지

Colab의 **런타임 → 런타임 유형 변경 → GPU**를 선택하고 위에서 아래로 실행하세요.
업로드할 파일: **components7_finetune_bundle.zip** 하나입니다.
ZIP 안에 기존 Colab 학습 `best_components7.pt`와 동일한 7클래스 데이터가 들어 있습니다.

- 공식 초기 모델에서 재학습하는 것이 아니라 **현재 best.pt의 가중치에서 이어서 학습**합니다.
- 새 효과와 낮은 학습률을 적용하려고 **resume=False**로 새 학습을 시작합니다.
- USB/빨간 LED 사진은 추가하지 않았습니다. 기존 분할 243 train / 63 val을 유지합니다.
- 검증 사진은 학습에 넣지 않습니다. 별도의 320×240 품질 평가 세트도 검증 사진에서만 만듭니다.
- 원본 가중치를 보관하고, 추가 학습 결과를 새 파일로 받습니다. 개선은 보장되지 않습니다.

## 선택한 증강

훈련 영상의 50%에는 아래 품질 저하 중 **최대 한 종류만** 적용합니다.
나머지 50%에는 이 품질 저하를 적용하지 않습니다. 밝기/대비와 기하 변환은 별도로 적용됩니다.

| 효과 | 전체 훈련 이미지 기준 확률 | 강도 |
|---|---:|---|
| 해상도 손실 | 25% | 가로·세로를 45~75%로 줄였다 복원 |
| 약한 Gaussian blur | 5% | 3×3, 작은 sigma |
| 약한 Motion blur | 5% | 3픽셀 커널 |
| JPEG 압축 | 10% | quality 75~95 |
| 약한 노이즈 | 5% | 표준편차 약 1~4/255 |
| 밝기/대비 | 별도 20% | ±10% |

좌우 반전 50%, 상하 반전 20%, 회전 ±20도, 약한 위치/크기 변화도 유지합니다.
Mosaic은 10%로 줄여 작은 LED가 지나치게 작아지는 것을 제한합니다.
**흑백, 색조 변경, RGB 채널 뒤집기, MixUp은 사용하지 않습니다.**
색상별 LED 구분을 보존하려고 기본 Albumentations의 ToGray도 사용자 정의 정책으로 대체합니다.


In [ ]:
EXTRA_EPOCHS = 40  # @param {type:"integer"}
IMGSZ = 640  # @param [416, 640] {type:"raw"}
BATCH = 16
LEARNING_RATE = 0.0001
PATIENCE = 12
SEED = 44
SAVE_TO_DRIVE = False  # @param {type:"boolean"}
assert EXTRA_EPOCHS > 0 and IMGSZ in (416, 640)
print("현재 가중치에서 새 설정으로 추가 학습:", EXTRA_EPOCHS, "epochs")
print("Drive 저장을 끄면 종료 전에 다운로드가 필수입니다.")


In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ["YOLO_CONFIG_DIR"] = "/content/yolo_finetune_config"
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "ultralytics==8.4.183", "albumentations==2.0.8", "pandas"])
import torch
import albumentations as A
import cv2
from ultralytics import YOLO
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "Albumentations:", A.__version__)


In [ ]:
from google.colab import files
import zipfile, tempfile, json, hashlib
print("components7_finetune_bundle.zip을 선택하세요. PT 파일을 별도로 올릴 필요 없습니다.")
uploaded = files.upload()
zips = [n for n in uploaded if n.lower().endswith(".zip")]
assert len(zips) == 1, "추가 학습용 ZIP 하나만 선택하세요."
EXTRACT_ROOT = Path(tempfile.mkdtemp(prefix="components7_ft_", dir="/content"))
with zipfile.ZipFile(zips[0]) as z:
    assert len(z.infolist()) < 10000
    for info in z.infolist():
        assert (EXTRACT_ROOT / info.filename).resolve().is_relative_to(EXTRACT_ROOT.resolve())
    z.extractall(EXTRACT_ROOT)
del uploaded
BUNDLE = EXTRACT_ROOT / "components7_finetune"
DATASET = BUNDLE / "components_dataset"
BASE_WEIGHTS = BUNDLE / "baseline_weights/best_components7.pt"
assert BASE_WEIGHTS.is_file() and (DATASET / "data.yaml").is_file()
metadata = json.loads((BUNDLE / "bundle_info.json").read_text())
assert hashlib.sha256(BASE_WEIGHTS.read_bytes()).hexdigest() == metadata["baseline_sha256"]
sys.path.insert(0, str(BUNDLE))
from fine_tune_policy import quality_transforms
EXPECTED_CLASSES = ["DHT11", "bluetooth", "led_green", "led_red", "led_yellow", "rgb", "usb"]
check_model = YOLO(str(BASE_WEIGHTS))
assert check_model.task == "detect"
assert list(check_model.names.values()) == EXPECTED_CLASSES, "현재 7클래스 모델인지 확인하세요."
print("현재 모델 확인:", check_model.names)
del check_model


In [ ]:
import json
import yaml
import hashlib
from collections import Counter, defaultdict
from PIL import Image

EXPECTED_CLASSES = ["DHT11", "bluetooth", "led_green", "led_red", "led_yellow", "rgb", "usb"]
config = yaml.safe_load((DATASET / "data.yaml").read_text())
assert config["names"] == dict(enumerate(EXPECTED_CLASSES)), "클래스 ID/이름이 예상과 다릅니다."
# 원래 노트북 경로 대신 Colab의 실제 압축 해제 경로로 변경합니다.
config["path"] = str(DATASET)
(DATASET / "data.yaml").write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
DATA_YAML = str(DATASET / "data.yaml")
counts = {}
hashes = defaultdict(set)
for split in ("train", "val"):
    counts[split] = Counter()
    images = sorted((DATASET / "images" / split).glob("*.jpg"))
    assert images, f"{split} 이미지가 없습니다."
    labels = list((DATASET / "labels" / split).glob("*.txt"))
    assert len(images) == len(labels), "이미지/라벨 수가 다릅니다."
    for image_path in images:
        with Image.open(image_path) as image:
            image.verify()
        label = DATASET / "labels" / split / (image_path.stem + ".txt")
        assert label.is_file(), f"라벨 누락: {image_path}"
        rows = label.read_text().strip().splitlines()
        assert rows, "이 패키지의 채택 이미지에는 객체 라벨이 있어야 합니다."
        for row in rows:
            parts = row.split()
            assert len(parts) == 5
            class_id = int(parts[0])
            x, y, w, h = map(float, parts[1:])
            assert 0 <= class_id < 7
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
            assert x-w/2 >= -1e-6 and y-h/2 >= -1e-6
            assert x+w/2 <= 1+1e-6 and y+h/2 <= 1+1e-6
            counts[split][class_id] += 1
        hashes[hashlib.sha256(image_path.read_bytes()).hexdigest()].add(split)
    assert set(counts[split]) == set(range(7)), f"{split}에 일부 클래스가 없습니다."
assert all(len(splits) == 1 for splits in hashes.values()), "동일 이미지가 학습/검증에 중복됐습니다."
manifest = json.loads((DATASET / "manifest.json").read_text())
groups = defaultdict(set)
for item in manifest:
    if item["status"] == "included":
        assert item["reviewed"]
        groups[item["similarity_group"]].add(item["split"])
assert all(len(splits) == 1 for splits in groups.values())
for i, name in enumerate(EXPECTED_CLASSES):
    print(f"{i}: {name:12s} train={counts['train'][i]:3d} val={counts['val'][i]:3d}")
print("라벨/이미지 검사 통과")


In [ ]:
from datetime import datetime
RUN_NAME = f"components7_lowres_finetune_{datetime.now():%Y%m%d_%H%M%S}"
if SAVE_TO_DRIVE:
    from google.colab import drive
    # Drive 인증 오류가 나면 SAVE_TO_DRIVE=False로 바꾸고 이 셀을 다시 실행하세요.
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/component_yolo_runs")
else:
    PROJECT = Path("/content/component_finetune_runs")
PROJECT.mkdir(parents=True, exist_ok=True)
REPORT_DIR = PROJECT / (RUN_NAME + "_comparison")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("학습 결과:", PROJECT / RUN_NAME)
print("비교 보고서:", REPORT_DIR)


## 품질 변형 미리보기
사진의 좌표/크기는 그대로 유지하면서 세부 정보만 변화시킵니다.
너무 흐려서 몸체를 구분할 수 없다면 이 시점에 정책 강도를 낮추세요.
이 미리보기는 학습 사진에서 가져오며 검증 데이터에는 확률 증강을 적용하지 않습니다.


In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display
preview_path = next((DATASET / "images/train").glob("led_red_*.jpg"))
original_image = cv2.imread(str(preview_path))
transforms = [
    ("original", None),
    ("downscale 0.5", A.Downscale(scale_range=(0.5, 0.5),
       interpolation_pair={"downscale": cv2.INTER_AREA, "upscale": cv2.INTER_LINEAR}, p=1)),
    ("weak blur", A.GaussianBlur(blur_limit=(3, 3), sigma_limit=(0.5, 0.5), p=1)),
    ("motion blur", A.MotionBlur(blur_limit=(3, 3), p=1)),
    ("JPEG 80", A.ImageCompression(quality_range=(80, 80), p=1)),
    ("weak noise", A.GaussNoise(std_range=(0.01, 0.01), mean_range=(0, 0), p=1)),
]
preview = Image.new("RGB", (960, 520), "white")
draw = ImageDraw.Draw(preview)
for i, (name, transform) in enumerate(transforms):
    output = original_image if transform is None else transform(image=original_image.copy())["image"]
    assert output.shape == original_image.shape
    x, y = (i % 3) * 320, (i // 3) * 260
    preview.paste(Image.fromarray(cv2.cvtColor(output, cv2.COLOR_BGR2RGB)).resize((320, 240)), (x, y))
    draw.text((x + 4, y + 242), name, fill="black")
display(preview)
preview.save(REPORT_DIR / "augmentation_preview.jpg")
print(A.Compose(quality_transforms()))


In [ ]:
# 검증 이미지만 축소해 저해상도 평가용 사본을 만듭니다. 학습에는 절대 넣지 않습니다.
import shutil
import yaml
LOWRES = EXTRACT_ROOT / "lowres_evaluation"
(LOWRES / "images/val").mkdir(parents=True, exist_ok=True)
(LOWRES / "labels/val").mkdir(parents=True, exist_ok=True)
for image_path in sorted((DATASET / "images/val").glob("*.jpg")):
    image = cv2.imread(str(image_path))
    h, w = image.shape[:2]
    # 실제 320x240 입력을 모사하며 원본 가로세로 비율을 유지합니다.
    new_h = max(1, round(h * 320 / w))
    small = cv2.resize(image, (320, new_h), interpolation=cv2.INTER_AREA)
    target = LOWRES / "images/val" / image_path.name
    assert cv2.imwrite(str(target), small, [cv2.IMWRITE_JPEG_QUALITY, 90])
    shutil.copy2(DATASET / "labels/val" / (image_path.stem + ".txt"),
                 LOWRES / "labels/val" / (image_path.stem + ".txt"))
low_cfg = {"path": str(LOWRES), "train": str(DATASET / "images/train"),
           "val": "images/val", "names": dict(enumerate(EXPECTED_CLASSES))}
LOWRES_YAML = str(LOWRES / "data.yaml")
Path(LOWRES_YAML).write_text(yaml.safe_dump(low_cfg, sort_keys=False))
SCENARIOS = {"clean_640": (DATA_YAML, 640), "lowres_320": (LOWRES_YAML, 320)}
print("평가 세트:", SCENARIOS)


In [ ]:
def evaluate_checkpoint(weights, stage):
    detector = YOLO(str(weights))
    output = {}
    for name, (data_yaml, size) in SCENARIOS.items():
        metrics = detector.val(data=data_yaml, imgsz=size, batch=8, device=0,
                               workers=2, plots=False,
                               project=str(REPORT_DIR), name=stage + "_" + name,
                               exist_ok=True)
        class_rows = []
        for i, class_id in enumerate(metrics.box.ap_class_index):
            class_rows.append({"class": detector.names[int(class_id)],
                               "precision": float(metrics.box.p[i]),
                               "recall": float(metrics.box.r[i]),
                               "mAP50": float(metrics.box.ap50[i]),
                               "mAP50_95": float(metrics.box.ap[i])})
        output[name] = {"precision": float(metrics.box.mp),
                        "recall": float(metrics.box.mr),
                        "mAP50": float(metrics.box.map50),
                        "mAP50_95": float(metrics.box.map),
                        "per_class": class_rows}
    return output

# 추가 학습 전에 현재 모델의 기준 성능을 저장합니다.
BASELINE_RESULTS = evaluate_checkpoint(BASE_WEIGHTS, "before")
(REPORT_DIR / "before.json").write_text(json.dumps(BASELINE_RESULTS, indent=2))
print(json.dumps(BASELINE_RESULTS, indent=2))


## 추가 학습
기존 가중치에서 학습률 0.0001로 시작합니다. 이전 학습률 0.001보다 낮게 설정했습니다.
`resume=True`는 이전 학습 설정을 복원하므로 이번처럼 효과를 바꾸는 추가 학습에는 사용하지 않습니다.
검증은 원본 63장으로 수행합니다. 추가 생성된 검증 사본으로 모델을 학습하지 않습니다.


In [ ]:
finetuned = YOLO(str(BASE_WEIGHTS))
finetuned.train(
    data=DATA_YAML,
    epochs=EXTRA_EPOCHS, patience=PATIENCE, imgsz=IMGSZ,
    batch=BATCH, device=0, workers=2,
    project=str(PROJECT), name=RUN_NAME, exist_ok=False,
    resume=False, seed=SEED, deterministic=True,
    optimizer="AdamW", lr0=LEARNING_RATE, lrf=0.1,
    warmup_epochs=1.0, warmup_bias_lr=0.001,
    weight_decay=0.0005,
    augmentations=quality_transforms(),
    degrees=20, translate=0.08, scale=0.2,
    fliplr=0.5, flipud=0.2,
    hsv_h=0.0, hsv_s=0.1, hsv_v=0.15, bgr=0.0,
    mosaic=0.1, close_mosaic=10,
    mixup=0.0, cutmix=0.0, copy_paste=0.0,
    save=True, save_period=5, plots=True,
)
RUN_DIR = Path(finetuned.trainer.save_dir)
NEW_BEST = RUN_DIR / "weights/best.pt"
assert NEW_BEST.is_file()
print("추가 학습 후보 모델:", NEW_BEST)


In [ ]:
AFTER_RESULTS = evaluate_checkpoint(NEW_BEST, "after")
(REPORT_DIR / "after.json").write_text(json.dumps(AFTER_RESULTS, indent=2))
import pandas as pd
comparison = []
class_comparison = []
for scenario in SCENARIOS:
    for stage, results in [("before", BASELINE_RESULTS), ("after", AFTER_RESULTS)]:
        row = {"scenario": scenario, "stage": stage}
        row.update({k: round(results[scenario][k] * 100, 2)
                    for k in ("precision", "recall", "mAP50", "mAP50_95")})
        comparison.append(row)
        for c in results[scenario]["per_class"]:
            class_comparison.append({"scenario": scenario, "stage": stage, **c})
display(pd.DataFrame(comparison))
display(pd.DataFrame(class_comparison))
pd.DataFrame(comparison).to_csv(REPORT_DIR / "comparison_percent.csv", index=False)
pd.DataFrame(class_comparison).to_csv(REPORT_DIR / "per_class_comparison.csv", index=False)
print("동일 장면에서 만든 저화질 평가이며 독립 촬영 테스트는 아닙니다.")
print("lowres_320 개선 여부와 clean_640 성능 하락을 함께 보고 새 모델 사용을 결정하세요.")


In [ ]:
# 실제 저화질 검증 사진에서 전/후 결과를 시각 비교합니다.
old_detector = YOLO(str(BASE_WEIGHTS))
new_detector = YOLO(str(NEW_BEST))
for cls in ("usb", "led_red", "led_green", "led_yellow"):
    image_path = next((LOWRES / "images/val").glob(cls + "_*.jpg"))
    display_pair = Image.new("RGB", (640, 260), "white")
    draw = ImageDraw.Draw(display_pair)
    for i, (stage, detector) in enumerate([("before", old_detector), ("after", new_detector)]):
        result = detector.predict(str(image_path), imgsz=320, conf=0.35,
                                   device=0, verbose=False)[0]
        display_pair.paste(Image.fromarray(result.plot()[..., ::-1]).resize((320, 240)), (i * 320, 0))
        draw.text((i * 320 + 4, 242), stage + " / " + cls, fill="black")
    display(display_pair)
    display_pair.save(REPORT_DIR / (cls + "_before_after.jpg"))


In [ ]:
# 기존 모델을 자동으로 덮어쓰지 않고 새 후보와 비교 보고서를 다운로드합니다.
export_dir = Path("/content") / (RUN_NAME + "_export")
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(NEW_BEST, export_dir / "best_components7_finetuned.pt")
shutil.copy2(BASE_WEIGHTS, export_dir / "best_components7_original.pt")
shutil.copytree(REPORT_DIR, export_dir / "comparison", dirs_exist_ok=True)
shutil.copytree(RUN_DIR, export_dir / "training", dirs_exist_ok=True)
shutil.copy2(BUNDLE / "fine_tune_policy.py", export_dir / "fine_tune_policy.py")
files.download(str(export_dir / "best_components7_finetuned.pt"))
result_zip = shutil.make_archive(str(export_dir), "zip", root_dir=export_dir)
files.download(result_zip)
print("다운로드가 끝날 때까지 런타임을 종료하지 마세요.")


## 노트북 CPU에서 새 후보 적용
원본 `best_components7.pt`는 유지하고 새 파일을 `yolo_training/`에 넣으세요.

```bash
source /opt/ros/humble/setup.bash
export ROS_DOMAIN_ID=44
cd ~/turtlebot3_ws/yolo_training
.venv/bin/python infer_ros.py --ros-args \
  -p weights:=/home/kim/turtlebot3_ws/yolo_training/best_components7_finetuned.pt \
  -p input_topic:=/csi/image_raw \
  -p imgsz:=320
```

학습이 끝난 모델의 추론에는 별도 증강을 적용하지 않습니다. 모델 구조도 기존 YOLO11n 그대로입니다.
효과가 나빠지면 `weights`를 기존 `best_components7.pt`로 바꾸면 됩니다.
이 설정만으로 USB와 빨간 LED 혼동이 해결되지는 않을 수 있습니다. 실제 오검출 사진 보강은 다음 단계입니다.

[Ultralytics 사용자 정의 증강](https://docs.ultralytics.com/integrations/albumentations/) ·
[Downscale 설명](https://albumentations.ai/explore/transform/Downscale/docs/)
